# TGA: Mettler Toledo E2290

`TGAData` holds temperature (K) and mass (mg). A blank run under the same
programme can be subtracted on load. The files are the test fixtures under
`tests/tga/e2290/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lab_data_parser as ldp

In [ ]:
TGA = Path("..") / "tests" / "tga" / "e2290"
sample_file = next(TGA.glob("*_sample_*.txt"))
blank_file = next(TGA.glob("*_blank_*.txt"))

tga_raw = ldp.TGAData.from_e2290(sample_file)
tga = ldp.TGAData.from_e2290(sample_file, baseline_path=blank_file)
print("initial mass / mg:", tga_raw.mass_init)

## Mass loss and its derivative

`derivative` is dm/dT; `smooth` applies a Savitzky-Golay filter first. The
isothermal start is cut off with `cut_front` (temperature in K), since dm/dT is
not defined while the temperature stands still.

In [ ]:
smoothed = tga.cut_front(temperature=273.15 + 50).smooth(window_length=21)
fig, (ax_m, ax_d) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
ax_m.plot(tga_raw.temperature - 273.15, tga_raw.mass, label="as measured")
ax_m.plot(tga.temperature - 273.15, tga.mass, label="blank subtracted")
ax_m.set_ylabel("mass / mg")
ax_m.legend()
ax_d.plot(smoothed.temperature - 273.15, smoothed.derivative)
ax_d.set_ylabel("dm/dT / mg K^-1")
ax_d.set_xlabel("temperature / °C")
plt.show()

## Relative mass

`mass_fraction` divides by the initial mass from the file header.

In [ ]:
if tga_raw.mass_init:
    fraction = tga_raw.mass_fraction
    print(f"mass left at {tga_raw.temperature[-1] - 273.15:.0f} °C: {100 * fraction[-1]:.1f} %")